# Self-contained KG pipeline

Runs entity extraction, relation extraction, and SEMHASH deduplication. Output is written to `output/graph.json` only.

In [ ]:
# Optional: run once to install dependencies
# %pip install -r requirements.txt

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

load_dotenv()

from lib.pipeline import (
    Graph,
    ModelConfig,
    get_entities,
    get_relations,
    run_semhash_deduplication,
)

cfg = ModelConfig(
    model=os.environ["LLM_MODEL"],
    api_key=os.environ.get("LLM_API_KEY"),
    api_base=os.environ.get("LLM_API_BASE"),
    temperature=float(os.environ.get("LLM_TEMPERATURE", "0.0")),
    reasoning_effort=os.environ.get("LLM_REASONING_EFFORT") or None,
    ssl_verify=os.environ.get("LLM_SSL_VERIFY", "true").lower()
    not in ("false", "0", "no"),
)

semhash_model = os.environ.get("SEMHASH_MODEL") or None

In [ ]:
text = Path("sample.txt").read_text(encoding="utf-8")
CONTEXT = "Family relationships"

In [ ]:
entities, _ = get_entities(text, model_config=cfg)
relations, _ = get_relations(text, entities, context=CONTEXT, model_config=cfg)

graph = Graph(
    entities=set(entities),
    relations=set(relations),
    edges={relation[1] for relation in relations},
)

graph = run_semhash_deduplication(graph, semhash_model=semhash_model)

In [ ]:
out = Path("output/graph.json")
out.parent.mkdir(exist_ok=True)
out.write_text(graph.model_dump_json(indent=2), encoding="utf-8")
print(f"Wrote {out}")